# Figure 5C: implicit vs explicit correction RMSE, delta-22 vs the natural-products test set

Bootstrap RMSE of implicit (PCM) vs explicit (OpenMM + vibrations) corrections, delta-22 vs the pooled complex/natural-product test set, chloroform and benzene (bars = mean, error bars = 2.5-97.5 percentile). Produces the ¹H panel (saved) and a ¹³C companion (inline only).

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/applications", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
from applications_reader import Applications
import applications
import applications_plots
import paths

In [ ]:
APPLICATIONS_HDF5 = paths.dataset_file("applications", root=REPO)
XLSX = os.path.join(REPO, "data", "applications", "applications_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
loader = Applications(APPLICATIONS_HDF5, XLSX)
query_df_nn = applications.build_query_df_nn(loader)
seed = applications.build_bootstrap_seed_coeffs(loader)

# pooled "Test Set" bootstrap RMSE distributions (one bin over all 13 complex molecules)
grouped = {}
for nuc in ["H", "C"]:
    preds = applications.apply_bootstrap_params_to_full_dataset(query_df_nn, seed[nuc], nucleus=nuc)
    grouped[nuc] = applications.compute_grouped_rmse(preds, applications.ALL_IN_ONE_BIN)

In [ ]:
applications_plots.plot_nps_benefit_barplot(
    loader.rmse_distribution("H"), grouped["H"], nucleus="H",
    solvents=[["chloroform"], ["benzene"]], np_solute_groups=applications.ALL_IN_ONE_BIN,
    formulas=["stationary_plus_pcm", "stationary_plus_qcd + openMM"],
    labels=["Implicit Solvent (SotA)", "Explicit Solvent + Vibrations (OpenMM)"],
    colors=["#A72608", "#61a89a"], formula_remap=applications.FORMULA_REMAP,
    figsize=(6, 5), y_min=0.0, y_max=0.37, save_path=figure_path("fig5c_benefit_1H.png"))

applications_plots.plot_nps_benefit_barplot(
    loader.rmse_distribution("C"), grouped["C"], nucleus="C",
    solvents=[["chloroform"], ["benzene"]], np_solute_groups=applications.ALL_IN_ONE_BIN,
    formulas=["stationary_plus_pcm", "stationary_plus_op_vib + openMM"],
    labels=["Implicit Solvent (SotA)", "Explicit Solvent + Vibrations (OpenMM)"],
    colors=["#A72608", "#61a89a"], formula_remap=applications.FORMULA_REMAP,
    figsize=(6, 5), y_min=0.0, save_path=None)   # inline-only companion, not saved to disk